[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Decorators &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

Run the cell below first.


In [1]:
import functools
import contextlib

print("ready")


ready


**1.** `@upper`, applied with `@` and by assignment.


In [2]:
def upper(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper


@upper
def label(name):
    return f"station {name}"


def label_by_hand(name):
    return f"station {name}"


label_by_hand = upper(label_by_hand)

print(label("Tromso"))
print(label_by_hand("Tromso"))


STATION TROMSO
STATION TROMSO


The two outputs match because the two forms are the same operation. `@upper` is
`label = upper(label)`, written where a reader will see it.


**2.** `@trace`, which reports on the way in and the way out.


In [3]:
def trace(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        shown = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        print(f"-> {func.__name__}({', '.join(shown)})")
        result = func(*args, **kwargs)
        print(f"<- {func.__name__} returned {result!r}")
        return result
    return wrapper


@trace
def difference(a, b):
    return round(a - b, 1)


difference(-2.6, -4.1)
print("name:", difference.__name__)


-> difference(-2.6, -4.1)
<- difference returned 1.5
name: difference


`difference.__name__` is `difference` rather than `wrapper` only because of `functools.wraps`.
Remove that line and run it again to see the difference.


**3.** `@count_calls`, which keeps a count on the function.


In [4]:
def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper


@count_calls
def reading():
    return -4.1


for _ in range(3):
    reading()

print("calls:", reading.calls)


calls: 3


A function is an object, so it can carry attributes like any other. `wrapper.calls` is set once, in
the decorator, and each call adds one. Because the decorated name holds `wrapper`, `reading.calls`
reads it back.

The count lives on this function alone. A second function decorated with `@count_calls` gets its own
`wrapper`, and so its own count.


**4.** `@clamp(low, high)`, a decorator that takes settings.


In [5]:
def clamp(low, high):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return max(low, min(high, func(*args, **kwargs)))
        return wrapper
    return decorator


@clamp(-90, 60)
def faulty():
    return 999


@clamp(-90, 60)
def frozen():
    return -120


print(faulty(), frozen())


60 -90


Three layers, as `retry` had: `clamp` holds the settings, `decorator` receives the function, and
`wrapper` runs on each call. `min(high, value)` caps it from above and `max(low, ...)` from below.


**5.** The same two decorators, in both orders.


In [6]:
@trace
@clamp(-90, 60)
def spike_traced_outside():
    return 999


@clamp(-90, 60)
@trace
def spike_traced_inside():
    return 999


print("final:", spike_traced_outside())
print()
print("final:", spike_traced_inside())

# With trace outside, it wraps the clamped function, so the value it sees
# coming back has already been clamped to 60. With trace inside, it wraps the
# raw function and reports 999; clamp then changes it to 60 on the way out.
# The final value is 60 either way. What differs is which value trace saw.


-> spike_traced_outside()
<- spike_traced_outside returned 60
final: 60

-> spike_traced_inside()
<- spike_traced_inside returned 999
final: 60


Both versions return `60`, and a test checking only the final value would not tell them apart. The
difference is in what the inner layer can see, which is exactly the difference the closing example
in the notebook showed with logging and retrying.


**6.** `TemporaryUnit` as a generator.


In [7]:
class Station:
    def __init__(self, name, unit="C"):
        self.name = name
        self.unit = unit


@contextlib.contextmanager
def temporary_unit(station, unit):
    previous = station.unit
    station.unit = unit
    try:
        yield station
    finally:
        station.unit = previous


north = Station("Tromso")

with temporary_unit(north, "F") as station:
    print("inside:        ", station.unit)
print("after:         ", north.unit)

try:
    with temporary_unit(north, "K"):
        print("inside:        ", north.unit)
        raise RuntimeError("stopped halfway")
except RuntimeError:
    pass
print("after an error:", north.unit)


inside:         F
after:          C
inside:         K
after an error: C


The class version needed `__init__`, `__enter__` and `__exit__`. This needs one function, and
`previous` is an ordinary local variable, where the class had to store it on `self` to carry it from
`__enter__` to `__exit__`.

The `finally` is what makes the restore happen after the error. Without it, the exception would be
raised at the `yield` and the line after it would never run.


---

&#8592; **Back to:** [Decorators](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/06-decorators.ipynb)  &nbsp;&middot;&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
